# RAG arm -- Qwen2.5-Coder inference with retrieved few-shot examples + retrieved schema

Companion notebook to `mongodb_nl_to_sql_1.ipynb` (the baseline, full-schema arm). Everything about model loading, generation parameters, and output post-processing is **identical** to that notebook on purpose -- the only thing that differs between the two arms is prompt *content* (retrieved few-shot examples + one database's schema here, vs. the full 6-database schema there). That keeps the comparison isolated to what retrieval adds, not confounded by a different model config.

**Before running this notebook**, the retrieval step must already have produced `rag/data/rag_prompts.json` -- run these locally first (CPU only, no GPU needed):
```
python rag/build_split.py
python rag/build_retrieval_index.py
python rag/build_prompts.py
```
then commit/push so this notebook's `git clone` picks up `rag/data/rag_prompts.json`.


In [ ]:
!git clone --branch evaluation-pipeline https://github.com/tarun1125/CSAIML-Capstone-Project-20.git


In [ ]:
%pip install transformers accelerate torch bitsandbytes sentencepiece pymongo huggingface_hub codebleu bert-score tree-sitter tree-sitter-python


In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()


In [ ]:
cd /content/CSAIML-Capstone-Project-20


In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("capstone-eval-rag")


## RAG inference

Loads `rag/data/rag_prompts.json` (21 held-out test cases, each with a `system_prompt` already assembled by `rag/build_prompts.py` from retrieved few-shot examples + the retrieved database's schema). No prompt construction happens in this cell -- retrieval already happened locally; this cell's only job is calling Qwen and saving raw output, same as the baseline notebook's cell 5.


In [ ]:
import json
import time
from pathlib import Path

# -----------------------------------------------------------------------------
# Load the pre-built RAG prompts (produced locally by rag/build_prompts.py,
# committed to the repo, pulled in by the git clone in cell 0). Each entry
# already carries its own system_prompt -- this cell does zero prompt
# construction, only generation, so behavior here matches cell 5 of the
# baseline notebook exactly aside from where the prompt came from.
#
# TOP_K here must match whatever K value you built rag_prompts*.json with
# (rag/build_prompts.py <K>) -- it only selects which already-built prompts
# file to load; it does NOT change how many examples are retrieved (that
# was already decided when build_prompts.py ran, locally, before this
# notebook exists). Set it, then Run All -- for a K-sweep, change this one
# value and re-run this cell for each of K=3, K=5, K=10.
# -----------------------------------------------------------------------------
TOP_K = 10  # <-- change to 3 or 5 for a K-sweep run, matching a prompts file
            #     you already built+pushed with `python rag/build_prompts.py <K>`

ROOT = Path.cwd()
if not (ROOT / "rag" / "data" / "rag_prompts.json").exists():
    ROOT = ROOT.parent

_suffix = "" if TOP_K == 10 else f"_k{TOP_K}"
PROMPTS_FILE = ROOT / "rag" / "data" / f"rag_prompts{_suffix}.json"

if not PROMPTS_FILE.exists():
    raise FileNotFoundError(
        f"{PROMPTS_FILE} not found -- run, locally (not in this notebook): "
        f"rag/build_split.py, rag/build_retrieval_index.py, then "
        f"`python rag/build_prompts.py{'' if TOP_K == 10 else ' ' + str(TOP_K)}`, "
        f"then commit+push {PROMPTS_FILE.name} before re-running this notebook."
    )

with open(PROMPTS_FILE, encoding="utf-8") as f:
    CASES = json.load(f)

log.info("Loaded %d RAG test cases from %s", len(CASES), PROMPTS_FILE)

predictions_rag = []

for i, case in enumerate(CASES, 1):
    qid = case["id"]
    nl = case["question"]
    database = case.get("gold_database")
    system_prompt = case["system_prompt"]

    log.info("[%d/%d] [%s] querying Qwen (RAG arm, predicted_db=%s, db_match=%s)...",
             i, len(CASES), qid, case.get("predicted_database"), case.get("database_match"))
    t0 = time.time()

    try:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": nl}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False  # greedy decoding -- temperature is a no-op here, left out on purpose
        )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
        raw = raw.replace("```python", "").replace("```", "").strip()

        latency = time.time() - t0
        log.info("[%s] OK (%.2fs): %s", qid, latency, raw[:80])

        predictions_rag.append({
            "id": qid,
            "question": nl,
            "database": database,
            "generated_query": raw,
            "latency_s": round(latency, 3),
            "predicted_database": case.get("predicted_database"),
            "database_match": case.get("database_match"),
            "retrieved_ids": case.get("retrieved_ids"),
        })

    except Exception as e:
        log.error("[%s] FAILED: %s", qid, e)
        predictions_rag.append({
            "id": qid,
            "question": nl,
            "database": database,
            "generated_query": "",
            "error": str(e)
        })

# Save to rag/data/qwen_rag_results{_suffix}.json -- same record shape
# (id/question/database/generated_query) as data/qwen2.5-coder_results.json,
# so the existing normalize.py works unchanged. The _suffix keeps a K=3/K=5
# sweep run from overwriting the K=10 results this notebook produced before:
#   python normalize.py rag/data/qwen_rag_results{_suffix}.json rag/data/qwen_rag_normalized{_suffix}.json
OUT_PATH = ROOT / "rag" / "data" / f"qwen_rag_results{_suffix}.json"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions_rag, f, indent=2)

log.info("Saved %d predictions -> %s", len(predictions_rag), OUT_PATH)


## Next steps (run locally, not in Colab)

Download `rag/data/qwen_rag_results.json` (or `qwen_rag_results_k3.json` / `_k5.json` for a K-sweep run) from this Colab session, then:
```
python normalize.py rag/data/qwen_rag_results.json rag/data/qwen_rag_normalized.json
python rag/score_rag.py
```
`rag/score_rag.py` needs a real Atlas connection so it runs locally, same as `execute_gold.py` / `evaluation/execute_queries.py` already do. It prints RAG's accuracy next to the baseline's accuracy recomputed on the *same* held-out test ids -- that same-slice comparison is the number that actually answers whether retrieval helped.

### K-sweep (K=3 / K=5 / K=10)

`build_prompts.py`'s `TOP_K` is a CLI arg now, not a hardcoded constant, so K=3/K=5/K=10 can be compared on the exact same 61-case test split instead of confounding K with dataset size the way earlier K changes did. For each K value you want to compare, locally:
```
python rag/build_prompts.py 3          # writes rag_prompts_k3.json
```
commit+push `rag_prompts_k3.json`, then in this notebook set `TOP_K = 3` in the generation cell above and re-run it (produces `qwen_rag_results_k3.json`), then locally:
```
python normalize.py rag/data/qwen_rag_results_k3.json rag/data/qwen_rag_normalized_k3.json
python rag/score_rag.py 3              # writes rag_vs_baseline_scores_k3.csv
```
Repeat with `5` for K=5. K=10 stays the original, unsuffixed filenames (`rag_prompts.json` / `qwen_rag_results.json` / ... / `rag_vs_baseline_scores.csv`) so nothing already using those breaks.

## Visualize RAG vs Baseline (run locally, after rag/score_rag.py)

Reads `rag/outputs/rag_vs_baseline_scores.csv` (produced by `rag/score_rag.py`) and plots two figures into `rag/outputs/figures/`: execution accuracy (Baseline test-slice vs RAG) and a funnel view (database retrieval quality vs final correctness) that shows whether the RAG pipeline is losing points in retrieval or in generation.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Robust root-detection -- works whether this cell runs from the repo root
# or from inside rag/ (this notebook's own directory), since score_rag.py
# is meant to be run locally, not in this Colab session.
ROOT = Path.cwd()
for _ in range(3):
    if (ROOT / "rag" / "outputs").exists():
        break
    ROOT = ROOT.parent

RAG_OUTPUT = ROOT / "rag" / "outputs"
RAG_FIGURES = RAG_OUTPUT / "figures"
RAG_FIGURES.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------
# Load data -- reads rag/outputs/rag_vs_baseline_scores.csv, produced by
# rag/score_rag.py. Fail loud if it's missing rather than silently plotting
# stale numbers.
# ---------------------------------------------------
scores_path = RAG_OUTPUT / "rag_vs_baseline_scores.csv"
if not scores_path.exists():
    raise FileNotFoundError(f"{scores_path} missing -- run rag/score_rag.py first")

rag_scores = pd.read_csv(scores_path)
print(rag_scores)

baseline_row = rag_scores[rag_scores["Arm"] == "Qwen-Baseline(test-slice)"].iloc[0]
rag_row = rag_scores[rag_scores["Arm"] == "Qwen-RAG"].iloc[0]
retrieval_row = rag_scores[rag_scores["Arm"] == "DatabaseRetrievalDiagnostic"].iloc[0]

delta = rag_row["Accuracy"] - baseline_row["Accuracy"]

# ---------------------------------------------------
# Figure 1: Execution Accuracy -- Baseline (test-slice) vs RAG
# Same held-out ids, same gold, same scoring code -- so this delta is purely
# the effect of retrieval-augmented few-shot prompting, not a different test
# set (see rag/score_rag.py's docstring for why that apples-to-apples
# slicing matters).
# ---------------------------------------------------
plt.figure(figsize=(6, 4))
bars = plt.bar(
    ["Qwen-Baseline\n(test-slice)", "Qwen-RAG"],
    [baseline_row["Accuracy"], rag_row["Accuracy"]],
    color=["#888888", "#2a7de1"],
)
plt.ylabel("Execution Accuracy (%)")
plt.title(f"RAG vs Baseline Execution Accuracy (n={int(baseline_row['Total'])}, Δ={delta:+.1f}pp)")
plt.ylim(0, 100)
for bar, row in zip(bars, [baseline_row, rag_row]):
    plt.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
        f"{row['Accuracy']:.1f}%\n({int(row['Correct'])}/{int(row['Total'])})",
        ha="center", va="bottom", fontsize=9,
    )
plt.tight_layout()
plt.savefig(RAG_FIGURES / "rag_vs_baseline_accuracy.png", dpi=300)
plt.close()

# ---------------------------------------------------
# Figure 2: Retrieval diagnostic vs final accuracy -- shows WHERE the RAG
# pipeline is losing points. Retrieval finds the right database almost every
# time; the gap down to the final accuracy is downstream, in query
# generation given the retrieved context -- not in retrieval itself.
# ---------------------------------------------------
plt.figure(figsize=(6, 4))
stages = ["DB Retrieval\n(right database?)", "RAG Final\n(correct answer?)"]
vals = [retrieval_row["Accuracy"], rag_row["Accuracy"]]
counts = [(retrieval_row["Correct"], retrieval_row["Total"]), (rag_row["Correct"], rag_row["Total"])]
bars = plt.bar(stages, vals, color=["#f0a500", "#2a7de1"])
plt.ylabel("Accuracy (%)")
plt.title("RAG Funnel: Retrieval Quality vs Final Correctness")
plt.ylim(0, 100)
for bar, v, (c, t) in zip(bars, vals, counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
        f"{v:.1f}%\n({int(c)}/{int(t)})",
        ha="center", va="bottom", fontsize=9,
    )
plt.tight_layout()
plt.savefig(RAG_FIGURES / "rag_funnel_diagnostic.png", dpi=300)
plt.close()

print(f"\nSaved figures to {RAG_FIGURES}")
print(f"Delta: {delta:+.1f} percentage points "
      f"(Baseline {baseline_row['Accuracy']:.1f}% -> RAG {rag_row['Accuracy']:.1f}%)")


                           Arm  Correct  Total  Accuracy
0    Qwen-Baseline(test-slice)        3     61      4.92
1                     Qwen-RAG       15     61     24.59
2  DatabaseRetrievalDiagnostic       59     61     96.72

Saved figures to /Users/tarungudapati/Documents/ai-projects/capstone-project/CSAIML-Capstone-Project-20/rag/outputs/figures
Delta: +19.7 percentage points (Baseline 4.9% -> RAG 24.6%)


## K-sweep comparison: K=3 vs K=5 vs K=10 (run locally, after scoring all three)

Reads all three `rag/outputs/rag_vs_baseline_scores*.csv` files (produced by `python rag/score_rag.py 3`, `5`, and the default K=10 run) and plots RAG accuracy across K values against the fixed baseline test-slice number, plus the database-retrieval diagnostic per K. All three runs share the same 61-case test split, same gold, same scoring code -- the only thing that changes across bars is how many few-shot examples `rag/build_prompts.py` retrieved.

**Caveat carried over from this run**: these numbers reflect one Colab session's fp16 generation output, which this session confirmed is not bit-identical across separate Colab runtimes even with `do_sample=False` and unchanged prompts (see the project notes) -- treat differences of a point or two between K values with that in mind, not as proof one K is definitively better, though the pattern (more examples helping, up to a point) is consistent with what K=3/5/10 were chosen to test.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Same root-detection pattern as the cell above, kept independent so this
# cell can be run on its own without re-running the single-K viz cell first.
ROOT = Path.cwd()
for _ in range(3):
    if (ROOT / "rag" / "outputs").exists():
        break
    ROOT = ROOT.parent

RAG_OUTPUT = ROOT / "rag" / "outputs"
RAG_FIGURES = RAG_OUTPUT / "figures"
RAG_FIGURES.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------
# Load all three K runs -- fail loud (naming the missing K) rather than
# silently plotting whichever subset happened to exist.
# ---------------------------------------------------
K_FILES = {3: "rag_vs_baseline_scores_k3.csv", 5: "rag_vs_baseline_scores_k5.csv", 10: "rag_vs_baseline_scores.csv"}

rows = {}
for k, fname in K_FILES.items():
    path = RAG_OUTPUT / fname
    if not path.exists():
        raise FileNotFoundError(f"{path} missing -- run `python rag/score_rag.py{'' if k == 10 else ' ' + str(k)}` first")
    df = pd.read_csv(path)
    rows[k] = {
        "rag": df[df["Arm"] == "Qwen-RAG"].iloc[0],
        "baseline": df[df["Arm"] == "Qwen-Baseline(test-slice)"].iloc[0],
        "retrieval": df[df["Arm"] == "DatabaseRetrievalDiagnostic"].iloc[0],
    }

ks = sorted(rows.keys())
baseline_acc = rows[ks[0]]["baseline"]["Accuracy"]  # identical across K by construction -- same 61 ids, K-independent
n_total = int(rows[ks[0]]["baseline"]["Total"])

print("K-sweep summary (n=%d test cases per K):" % n_total)
for k in ks:
    r = rows[k]["rag"]
    print(f"  K={k:>2}: RAG {int(r['Correct'])}/{int(r['Total'])} ({r['Accuracy']:.1f}%)")
print(f"  Baseline (K-independent): {baseline_acc:.1f}%")

# ---------------------------------------------------
# Figure 3: RAG accuracy across K, with the fixed baseline as a reference line
# ---------------------------------------------------
plt.figure(figsize=(6, 4))
rag_accs = [rows[k]["rag"]["Accuracy"] for k in ks]
rag_counts = [(int(rows[k]["rag"]["Correct"]), int(rows[k]["rag"]["Total"])) for k in ks]
bars = plt.bar([f"K={k}" for k in ks], rag_accs, color="#2a7de1")
plt.axhline(baseline_acc, color="#888888", linestyle="--", linewidth=1.5,
            label=f"Baseline (test-slice, K-independent): {baseline_acc:.1f}%")
plt.ylabel("RAG Execution Accuracy (%)")
plt.title(f"RAG Accuracy vs K (few-shot examples retrieved), n={n_total}")
plt.ylim(0, max(rag_accs + [baseline_acc]) + 15)
plt.legend(loc="upper left", fontsize=8)
for bar, (c, t) in zip(bars, rag_counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
              f"{bar.get_height():.1f}%\n({c}/{t})", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(RAG_FIGURES / "rag_k_sweep_accuracy.png", dpi=300)
plt.close()

# ---------------------------------------------------
# Figure 4: retrieval diagnostic across K, alongside final accuracy -- shows
# whether a bigger K changes which database gets predicted, not just how
# many examples get shown.
# ---------------------------------------------------
plt.figure(figsize=(6, 4))
x = range(len(ks))
width = 0.35
retrieval_accs = [rows[k]["retrieval"]["Accuracy"] for k in ks]
plt.bar([i - width / 2 for i in x], retrieval_accs, width, color="#f0a500", label="DB Retrieval")
plt.bar([i + width / 2 for i in x], rag_accs, width, color="#2a7de1", label="RAG Final")
plt.xticks(list(x), [f"K={k}" for k in ks])
plt.ylabel("Accuracy (%)")
plt.title("Retrieval Quality vs Final Correctness, by K")
plt.ylim(0, 110)
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(RAG_FIGURES / "rag_k_sweep_funnel.png", dpi=300)
plt.close()

print(f"\nSaved K-sweep figures to {RAG_FIGURES}")


K-sweep summary (n=61 test cases per K):
  K= 3: RAG 13/61 (21.3%)
  K= 5: RAG 14/61 (22.9%)
  K=10: RAG 15/61 (24.6%)
  Baseline (K-independent): 4.9%

Saved K-sweep figures to /Users/tarungudapati/Documents/ai-projects/capstone-project/CSAIML-Capstone-Project-20/rag/outputs/figures
